In [ ]:
!pip install -q tiktoken
import pandas as pd
from google.colab import userdata
from openai import OpenAI
from datetime import datetime
import time
import tiktoken
import pandas as pd
import re

In [ ]:
#Prepare the datasets and subsets that we're going to use:
espontaneo = "/content/c_espontaneo.txt"
sintetico = "/content/c_sintetico.txt"
sintetico_all = "/content/c_sintetico_all.txt"


def extractlines(filename):
  with open(filename, 'r') as f:
    lines = [line.strip() for line in f.readlines()]

  pattern = re.compile(r'^[\wáéíóúÁÉÍÓÚüÜñÑ.\s]+:')

  # Initialize lists for filtered and removed lines
  filtered_lines = []
  removed_lines = []

  # Iterate through the original lines and apply the filter
  for line in lines:
      if pattern.match(line):
          filtered_lines.append(line)
      else:
          removed_lines.append(line)

  return filtered_lines


In [ ]:
espontaneo_lines=extractlines(espontaneo)
sintetico_lines=extractlines(sintetico)
sintetico_all_lines=extractlines(sintetico_all)

### 3.2 Calling the Model

This function sends the prompt to the OpenAI API and returns the response:

In [ ]:
client_gpt = OpenAI(api_key=userdata.get('OPENAI_API_KEY'))
## Calls the OpenAI API with the given prompt.
def call_model(prompt, model_name, model_client, temperature):
    response = model_client.chat.completions.create(
        model=model_name,
        temperature=temperature,
        messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content

test_sentences =sintetico_all_lines  #Deifne the dataset we're going to use

exptype="sintetico_all"

In [ ]:
# --- you may want to adjust these rates to your actual OpenAI plan ---
# Prices are expressed per 1,000,000 tokens
MODEL_PRICING = {
    "gpt-3.5-turbo": {"prompt": 1.5, "completion": 2.0},
    "gpt-4o": {"prompt": 2.5, "completion": 10.0},
    "gpt-5-mini": {"prompt": 1.0, "completion": 1.2}, # Placeholder pricing
    "llama3.3-70b": {"prompt": 2.8, "completion": 2.8},
    "llama3.1-70b": {"prompt": 2.8, "completion": 2.8},
    "llama4-maverick": {"prompt": 2.8, "completion": 2.8},
    "gpt-oss-120b": {"prompt": 2.8, "completion": 2.8}
}


# Calls the OpenAI API with the given prompt.
def call_model(prompt, model_name, model_client, temperature):
    response = model_client.chat.completions.create(
        model=model_name,
        temperature=temperature,
        messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content


def run_experiment(
    experiment_type,
    model_name,
    model_client,
    support_set=None,
    source_column="",
    target_column="",
    num_few_shot_examples=0,
    instruction_template="",
    temperature=0,
    num_retries=3,
    quiet=False
):
    """Runs a zero-shot or few-shot experiment, retries on API failures,
    auto-counts tokens for cost, and saves results & logs cost."""
    # prepare tokenizer for local token counting
    try:
        encoder = tiktoken.encoding_for_model(model_name)
    except KeyError:
        encoder = tiktoken.get_encoding("cl100k_base")  # fallback

    #test_sentences = IAmex["Titulo"].values
    #test_sentences = IAgranulada["Titulo"].values
    instruction = instruction_template
    results = []
    total_cost = 0.0


    for i, test_sentence in enumerate(test_sentences, 1):
        # construct prompt with dynamic labels
        prompt = f"{instruction}:\n {test_sentence}"
        #print(prompt)
        # retry logic for unstable API calls
        attempt = 0
        while True:
            try:
                response = call_model(
                    prompt,
                    model_name,
                    model_client,
                    temperature
                )
                break
            except Exception as e:
                attempt += 1
                if attempt >= num_retries:
                    print(f"Iteration {i}: Failed after {num_retries} attempts. Error: {e}")
                    raise
                wait_time = 2 ** (attempt - 1)
                print(f"API call failed (attempt {attempt}/{num_retries}), retrying in {wait_time}s...")
                time.sleep(wait_time)

        # extract generated text
        if isinstance(response, dict):
            try:
                result_text = response["choices"][0]["message"]["content"]
            except KeyError:
                result_text = response["choices"][0]["text"]
        else:
            result_text = response

        # local token counting for cost
        prompt_tokens = len(encoder.encode(prompt))
        result_tokens = len(encoder.encode(result_text))

        # compute cost
        rates = MODEL_PRICING.get(model_name, {})
        cost = (prompt_tokens / 1_000_000) * rates.get("prompt", 0.0) + \
               (result_tokens / 1_000_000) * rates.get("completion", 0.0)
        total_cost += cost
        results.append([test_sentence, result_text])

        if not quiet:
            print(
                f"Iteration {i}:\n"
                f"{test_sentence}\n"
                f"{result_text}\n"
                f"Cost = ${cost:.6f}\n"
            )
        else:
            print(f"\r{i}/{len(test_sentences)}", end="")

    # save results
    # Sanitize model_name for filename by replacing slashes
    sanitized_model_name = model_name.replace("/", "_") # Use replace to handle all slashes
    output_filename = f"{sanitized_model_name}_{experiment_type}"
    df_results = pd.DataFrame(results)
    out_path = f"{output_filename}.tsv"
    df_results.to_csv(out_path, sep="\t", index=False)
    print(f"\nExperiment results saved to '{out_path}'.")

    print(f"Total cost: ${total_cost:.6f} (logged in costs.txt)\n")

In [ ]:
models = [
    #("gpt-3.5-turbo", client_gpt),
    #("gpt-5-mini", client_gpt)
    ("gpt-5.1", client_gpt),
     #("llama3.1-70b", client_llama)#,
    #("llama3.3-70b", client_llama)
    #("llama4-maverick", client_llama)
    #("openai/gpt-oss-120b", client_hf)
    #("meta-llama/Meta-Llama-3-8B-Instruct", client_hf)
    #("deepseek-ai/DeepSeek-V3.2-Exp", client_hf)
    #("meta-llama/Meta-Llama-3.1-70B-Instruct", client_llama3)



]

for model, client in models:
    run_experiment(

        experiment_type=exptype,
        #experiment_type="NER-IA_Mexico_2023_2024_2025-headlines",
        model_name=model,
        model_client=client,
        instruction_template = """Perform Named Entity Recognition (NER) on the following Spanish text. Return only the NER results. Do not include explanations, comments, or any additional text.
        Return the results in a json format (just text and label)"""#Output format:
        #Entity | Label"""

        #,quiet = True
    )

Streaming output truncated to the last 5000 lines.
  {
    "text": "Ricardito",
    "label": "PER"
  },
  {
    "text": "Max",
    "label": "PER"
  },
  {
    "text": "Benito en América",
    "label": "WORK_OF_ART"
  }
]
Cost = $0.000000

Iteration 93:
Max: ¡Hola, hola, hola! Bienvenidos a "Benito en América", el único show donde los mariachis zombies se echan un palomazo mientras hablamos de cosas bien serias. Soy Max, su anfitrión, y hoy tenemos a una invitada muy especial, ¡una mujer mexicana que baila como si tuviera un motor en los pies! Pero cuidado, porque también es más conflictiva que un taco sin salsa. ¿Cómo estás hoy?
[
  {"text": "Max", "label": "PER"},
  {"text": "Benito en América", "label": "MISC"},
  {"text": "Max", "label": "PER"},
  {"text": "mexicana", "label": "NORP"}
]
Cost = $0.000000

Iteration 94:
Ricardito: ¡Ay, Max! Estoy más emocionado que un perro con dos colas. Aunque debo confesar que hoy ando más torpe que un elefante en una cristalería. ¿Sabes? Mi pedido